In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# BigQuery Agent Analytics — the four-guarantee decision-lineage demo

**You own the graph. The SDK validates it cheaply, extracts deterministically, and
resolves user inputs to canonical concepts.**

This notebook walks through the four guarantees the SDK ships post-V5:

| Guarantee | What it means | Beat |
|---|---|---|
| **Own** | The user authors `CREATE PROPERTY GRAPH`. The SDK populates base tables — it never rewrites your graph DDL. | 1 |
| **Validate** | A sub-second pre-flight catches binding/table drift before extraction spends a dollar. | 2 |
| **Extract cheaply** | Deterministic compiled extractors handle structured events; `AI.GENERATE` only fills semantic gaps. | 3 |
| **Resolve** | User-typed inputs (canonical entity label in this fixture; `skos:altLabel` / `skos:prefLabel` / `skos:notation` rows in richer ontologies) resolve via the SKOS concept index emitted alongside the property graph. | 4 |

The demo's domain is **MAKO** — the Monetization Agents Knowledge Ontology, a real
Yahoo Monetization Platform ontology covering audience-segment / bid-value / creative-
variant / frequency-cap decisions. The agent that produces the events is a real ADK
Gemini agent talking to the BQ AA plugin; the events you see below were not synthesized.

## Section 0 — what you need to bring

**Minimum hand-authored input: one ontology file.**

Everything else — `binding.yaml`, `table_ddl.sql`, the property graph, the trace
events — is either auto-generated by the SDK's CLI, owned by you (your tables, your
graph DDL), or emitted by the BQ AA plugin when an agent runs. Two YAML files is
the *common* shape, but only one is *required*.

Three input shapes are equivalent today:

| Input shape | What you author | What's generated |
|---|---|---|
| **(a) Hand-authored YAML** | `ontology.yaml` | `binding.yaml` (`gm scaffold`), `table_ddl.sql` |
| **(b) OWL/SKOS TTL** | `*.ttl` | `ontology.yaml` (`gm import-owl`), then (a) |
| **(c) Future `@builtin:adk-events`** | nothing | everything |

This notebook uses shape **(b)**: `examples/migration_v5/mako_core.ttl` is the
authored input. The TTL → ontology → binding → DDL pipeline lives in
`mako_artifacts.py` (a thin convenience wrapper around `gm import-owl` + `gm scaffold`
tuned for this demo's six-entity scope).

### Install + authenticate + configure

Install the SDK, the ontology package, and the BQ AA plugin. The agent uses Vertex
AI Gemini, so the runtime needs Vertex access on `PROJECT_ID` and BigQuery write
access on the same project.

In [2]:
# Environment-aware install.
#
# * Colab: ``%pip install`` is Jupyter magic; persists into
#   the kernel env without the shell-side glob issues that
#   make plain ``!pip install ... google-adk[vertexai] ...``
#   fail under zsh.
# * Local kernel: the install cell is intentionally a
#   no-op — system Python is PEP-668 managed and refuses
#   ``pip install`` outside a venv. The expected
#   workflow is to ``pip install`` into a venv first and
#   launch the kernel against it, so this cell just
#   verifies the packages the rest of the notebook needs
#   are importable and prints a clear hint if not.
try:
    import google.colab  # noqa: F401 — Colab detection
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    get_ipython().run_line_magic(
        "pip",
        'install -q bigquery-agent-analytics bigquery-ontology '
        '"google-adk[vertexai]" google-cloud-bigquery pyyaml '
        'rdflib python-dotenv',
    )
else:
    _required = (
        "bigquery_agent_analytics",
        "bigquery_ontology",
        "google.adk",
        "google.cloud.bigquery",
        "yaml",
        "rdflib",
        "dotenv",
    )
    import importlib
    _missing = [
        m for m in _required
        if importlib.util.find_spec(m) is None
    ]
    if _missing:
        raise RuntimeError(
            f"Missing packages: {_missing}. Install them into\n"
            "your venv before launching this kernel:\n\n"
            "    pip install bigquery-agent-analytics bigquery-ontology \\\n"
            "        \"google-adk[vertexai]\" google-cloud-bigquery \\\n"
            "        pyyaml rdflib python-dotenv"
        )
    print("Local kernel — all required packages are importable.")


Local kernel — all required packages are importable.


In [3]:
import os

try:
    from google.colab import auth as _colab_auth
    _colab_auth.authenticate_user()
    print("Colab authentication successful.")
except ImportError:
    print("Not running in Colab — using application-default credentials.")

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "your-project-id")
DATASET_BASE = os.environ.get("BQ_DATASET", "migration_v5_demo")
AGENT_LOCATION = os.environ.get("DEMO_AGENT_LOCATION", "us-central1")
DATASET_LOCATION = os.environ.get("DATASET_LOCATION", "US")

print(f"Project  : {PROJECT_ID}")
print(f"Agent location : {AGENT_LOCATION}")
print(f"Dataset location : {DATASET_LOCATION}")
assert PROJECT_ID != "your-project-id", (
    "Set GOOGLE_CLOUD_PROJECT before running this notebook."
)


Not running in Colab — using application-default credentials.
Project  : test-project-0728-467323
Agent location : us-central1
Dataset location : US


### Scratch dataset + feature flags

Every run creates a fresh `migration_v5_demo_<8-hex>` dataset with 1-hour table TTL,
so re-running the notebook never collides with a previous run.

`FEATURES` gates each guarantee's cells. Flip an entry to `False` to skip live,
expensive, or environment-specific beats (e.g. you want to read the storyboard
without running Vertex AI, or the cluster you're on has no BQ AI quota). All
underlying issues have shipped in the SDK; gated cells degrade to
`Skipped: feature off` markdown rather than failing.

In [4]:
import uuid

from google.cloud import bigquery

RUN_ID = uuid.uuid4().hex[:8]
DATASET_ID = f"{DATASET_BASE}_{RUN_ID}"

bq = bigquery.Client(project=PROJECT_ID, location=DATASET_LOCATION)
ds = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
ds.location = DATASET_LOCATION
ds.default_table_expiration_ms = 3600_000  # 1 hour
bq.create_dataset(ds, exists_ok=True)
print(f"Scratch dataset: {PROJECT_ID}.{DATASET_ID}  (TTL 1h, location {DATASET_LOCATION})")

FEATURES = {
    "skip_property_graph": True,    # #104
    "binding_validate": True,       # #105
    "validate_extracted_graph": True,  # #76
    "compiled_extractors_c1": True,  # #75 PR 4b/4c — compile + measurement
    "compiled_extractors_c2": True,  # #75 C2 — runtime bundle loading
    "concept_index_reader": True,    # #58 reader follow-on
}
print("FEATURES:")
for k, v in FEATURES.items():
    print(f"  {k:30s} {'ON' if v else 'off'}")


Scratch dataset: test-project-0728-467323.migration_v5_demo_56f9179f  (TTL 1h, location US)
FEATURES:
  skip_property_graph            ON
  binding_validate               ON
  validate_extracted_graph       ON
  compiled_extractors_c1         ON
  compiled_extractors_c2         ON
  concept_index_reader           ON


### Generate ontology + binding + DDL from the MAKO TTL

`mako_artifacts.regenerate_snapshots(project, dataset)` runs the equivalent of

```
gm import-owl mako_core.ttl --out ontology.yaml
gm scaffold --ontology ontology.yaml --project P --dataset D --out .
```

plus the demo-specific post-processing this notebook needs: drops cross-namespace
PROV-O / PKO relationships (the OWL importer can't resolve those endpoints), restricts
the binding to the six demo entities (`AgentSession`, `DecisionExecution`,
`DecisionPoint`, `Candidate`, `SelectionOutcome`, `ContextSnapshot`), and maps each
ontology property type to its BQ column type. The output is four files:

- `ontology.yaml` — 18 MAKO entities with primary keys resolved
- `binding.yaml` — 6 entities + 7 relationships against `{PROJECT_ID}.{DATASET_ID}`
- `table_ddl.sql` — `CREATE TABLE IF NOT EXISTS` for every node + edge table, including `session_id STRING, extracted_at TIMESTAMP` SDK metadata columns
- `property_graph.sql` — `CREATE OR REPLACE PROPERTY GRAPH` over the same tables

In [5]:
import sys
sys.path.insert(0, "examples/migration_v5")

import mako_artifacts

artifact_counts = mako_artifacts.regenerate_snapshots(
    project=PROJECT_ID,
    dataset=DATASET_ID,
)
print(artifact_counts)

from pathlib import Path
HERE = Path("examples/migration_v5")
ONTOLOGY_PATH = HERE / "ontology.yaml"
BINDING_PATH = HERE / "binding.yaml"
TABLE_DDL_PATH = HERE / "table_ddl.sql"
PROPERTY_GRAPH_PATH = HERE / "property_graph.sql"
print()
for p in (ONTOLOGY_PATH, BINDING_PATH, TABLE_DDL_PATH, PROPERTY_GRAPH_PATH):
    print(f"{p}  {p.stat().st_size:>6} bytes")


{'ontology_entities': 18, 'binding_entities': 6, 'binding_relationships': 7}

examples/migration_v5/ontology.yaml   10922 bytes
examples/migration_v5/binding.yaml    2928 bytes
examples/migration_v5/table_ddl.sql    2563 bytes
examples/migration_v5/property_graph.sql    3867 bytes


### Apply the table DDL

Run the `CREATE TABLE IF NOT EXISTS` block — fifteen statements, one per node + edge
table. The plugin's own `agent_events` table is created separately when the plugin
starts; this DDL just creates the MAKO graph tables that the SDK will materialize
into.

In [6]:
ddl = TABLE_DDL_PATH.read_text()
for stmt in ddl.split(";"):
    stmt = stmt.strip()
    if not stmt:
        continue
    bq.query(stmt + ";").result()
print(f"Applied {ddl.count(';')} DDL statements against {DATASET_ID}.")


Applied 13 DDL statements against migration_v5_demo_56f9179f.


### Run the MAKO agent — populate `agent_events`

`run_agent.py` drives a real ADK Gemini agent through `N` MAKO decision sessions.
Each session walks the canonical decision flow:

```
capture_context  →  propose_decision_point  →  evaluate_candidate (×3-5)  →  commit_outcome  →  complete_execution
```

The BQ AA plugin attached to the runner captures every invocation, agent, LLM, tool,
and HITL event into `{PROJECT_ID}.{DATASET_ID}.agent_events` as a real plugin trace.
Nothing is synthesized.

**This cell is the only one that costs Vertex tokens.** Subsequent cells read from the
table we just populated. Keep `--sessions` small (10-50) while iterating.

In [7]:
import subprocess

SESSIONS = int(os.environ.get("MIGRATION_V5_SESSIONS", "10"))
cmd = [
    sys.executable,
    "examples/migration_v5/run_agent.py",
    "--sessions", str(SESSIONS),
    "--project", PROJECT_ID,
    "--dataset", DATASET_ID,
    "--location", DATASET_LOCATION,
]
# Propagate AGENT_LOCATION into the child process so the
# notebook variable (not just the OS env) drives Vertex
# region. Same for PROJECT_ID / DATASET_ID, even though
# they're already on the command line, so the agent module
# can pick them up from env at import time too.
child_env = {
    **os.environ,
    "DEMO_AGENT_LOCATION": AGENT_LOCATION,
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "PROJECT_ID": PROJECT_ID,
    "DATASET_ID": DATASET_ID,
    "DATASET_LOCATION": DATASET_LOCATION,
}
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=child_env)


/usr/local/opt/python@3.13/bin/python3.13 examples/migration_v5/run_agent.py --sessions 3 --project test-project-0728-467323 --dataset migration_v5_demo_56f9179f --location US


Running 3 sessions against test-project-0728-467323.migration_v5_demo_56f9179f.agent_events
  session 0/3


/Users/haiyuancao/adk-python/src/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


done.


CompletedProcess(args=['/usr/local/opt/python@3.13/bin/python3.13', 'examples/migration_v5/run_agent.py', '--sessions', '3', '--project', 'test-project-0728-467323', '--dataset', 'migration_v5_demo_56f9179f', '--location', 'US'], returncode=0)

In [8]:
# Sanity check: count rows by event_type in agent_events.
rows = bq.query(f"""
    SELECT event_type, COUNT(*) AS n
    FROM `{PROJECT_ID}.{DATASET_ID}.agent_events`
    GROUP BY event_type
    ORDER BY n DESC
""").result()
for r in rows:
    print(f"  {r.event_type:30s} {r.n}")


  LLM_REQUEST                    27
  LLM_RESPONSE                   27
  TOOL_STARTING                  24
  TOOL_COMPLETED                 24
  INVOCATION_STARTING            3
  USER_MESSAGE_RECEIVED          3
  AGENT_STARTING                 3
  AGENT_COMPLETED                3
  INVOCATION_COMPLETED           3
  AGENT_RESPONSE                 1


Section 0 is done. We have:

- A scratch dataset with the MAKO node + edge tables created (empty).
- An `agent_events` table populated by a real ADK Gemini agent + BQ AA plugin.
- Authored `ontology.yaml` + `binding.yaml` + `property_graph.sql`.

The four-guarantee story starts in Section 1.

## Section 1 — Beat 1: you own the graph definition (#104)

**Guarantee:** Own.

Before #104, every `ontology-build` ran `CREATE OR REPLACE PROPERTY GRAPH`
unconditionally. If you'd hand-tuned the graph DDL (additional `LABEL` properties,
an extra edge table layered on top, view-style alias columns), the build silently
overwrote your changes on the next run.

After #104, the build accepts `--skip-property-graph` and the SDK takes the position
that the **graph object is yours**. The SDK extracts events, populates the base
tables, and leaves your `CREATE PROPERTY GRAPH` alone.

This section proves both halves: the SDK does not issue a `CREATE PROPERTY GRAPH`
job during the build, and the graph object you defined still answers GQL queries
after the build finishes.

### 1.2 — apply the user's authored property-graph DDL

`examples/migration_v5/property_graph.sql` is hand-authored DDL — the binding produced
the column shape (`from_columns` / `to_columns`), but the property-graph definition
itself is a fixture the platform team would own in their own repo. The notebook reads
it from disk and applies it to the scratch dataset.

In [9]:
if not FEATURES["skip_property_graph"]:
    print("Skipped: skip_property_graph feature off")
else:
    property_graph_ddl = PROPERTY_GRAPH_PATH.read_text()
    print(property_graph_ddl)
    bq.query(property_graph_ddl).result()
    print("Property graph applied.")


CREATE OR REPLACE PROPERTY GRAPH `test-project-0728-467323.migration_v5_demo_56f9179f.mako_demo_graph`
  NODE TABLES (
    `test-project-0728-467323.migration_v5_demo_56f9179f.agent_session` AS agent_session
      KEY (agent_session_id)
      LABEL AgentSession PROPERTIES (agent_session_id, session_id),
    `test-project-0728-467323.migration_v5_demo_56f9179f.candidate` AS candidate
      KEY (candidate_id)
      LABEL Candidate PROPERTIES (candidate_id),
    `test-project-0728-467323.migration_v5_demo_56f9179f.context_snapshot` AS context_snapshot
      KEY (context_snapshot_id)
      LABEL ContextSnapshot PROPERTIES (context_snapshot_id, snapshot_payload, snapshot_timestamp),
    `test-project-0728-467323.migration_v5_demo_56f9179f.decision_execution` AS decision_execution
      KEY (decision_execution_id)
      LABEL DecisionExecution PROPERTIES (decision_execution_id, business_entity_id, latency_ms, span_id, trace_id),
    `test-project-0728-467323.migration_v5_demo_56f9179f.decisi

Property graph applied.


### 1.3 — capture the before-build baseline

**Order matters here.** The timestamp must be captured *after* cell 1.2's
`CREATE PROPERTY GRAPH` job has finished and *before* cell 1.4 runs the build,
otherwise the `JOBS_BY_PROJECT` filter in cell 1.5 catches the user's own authored
DDL job and produces a false positive.

The GQL traversal returns zero rows right now (the base tables are empty until the
build runs in 1.4). That's expected — the point of this cell is to prove the graph
object *executes* against the user's DDL, not that any rows exist yet.

In [10]:
if not FEATURES["skip_property_graph"]:
    before_skip_build_ts = None
    before_skip_build_rows = None
    print("Skipped: skip_property_graph feature off")
else:
    ts_row = next(iter(bq.query(
        "SELECT CURRENT_TIMESTAMP() AS ts"
    ).result()))
    before_skip_build_ts = ts_row.ts
    print(f"before_skip_build_ts = {before_skip_build_ts.isoformat()}")

    gql = f"""
        SELECT COUNT(*) AS n
        FROM GRAPH_TABLE(
          `{PROJECT_ID}.{DATASET_ID}.mako_demo_graph`
          MATCH (de:DecisionExecution)
          COLUMNS (de.decision_execution_id AS execution_id)
        )
    """
    before_skip_build_rows = next(iter(bq.query(gql).result())).n
    print(f"GQL DecisionExecution count (before build) = {before_skip_build_rows}")


before_skip_build_ts = 2026-05-14T16:05:18.390378+00:00


GQL DecisionExecution count (before build) = 0


### 1.4 — run `ontology-build --skip-property-graph`

Discover the session IDs the agent produced (the BQ AA plugin tags every event with
the ADK-generated session ID), then run the build with `--skip-property-graph` so the
SDK populates base tables but **does not** rewrite the property graph.

Note the absence of any `--graph-name` override: the SDK can't tell which graph the
user intends to populate when `--skip-property-graph` is set, because the build's job
is to populate base tables, not to maintain the graph object. The user's DDL — applied
in cell 1.2 — is the source of truth for the graph definition.

In [11]:
if not FEATURES["skip_property_graph"]:
    session_ids = []
    build_result = None
    print("Skipped: skip_property_graph feature off")
else:
    session_ids = [
        row.session_id for row in bq.query(f"""
            SELECT DISTINCT session_id
            FROM `{PROJECT_ID}.{DATASET_ID}.agent_events`
            WHERE session_id IS NOT NULL
            ORDER BY session_id
        """).result()
    ]
    print(f"Discovered {len(session_ids)} sessions in agent_events")
    assert session_ids, "agent_events has no sessions — re-run Section 0."

    import json as _json
    build_cmd = [
        sys.executable, "-m", "bigquery_agent_analytics.cli",
        "ontology-build",
        "--project-id", PROJECT_ID,
        "--dataset-id", DATASET_ID,
        "--ontology", str(ONTOLOGY_PATH),
        "--binding", str(BINDING_PATH),
        "--session-ids", ",".join(session_ids),
        "--skip-property-graph",
        "--location", DATASET_LOCATION,
        "--format", "json",
    ]
    print(" ".join(build_cmd))
    # ``capture_output=True`` masks stderr from the notebook;
    # surface it on failure so the operator sees the actual
    # error without re-running the CLI by hand.
    proc = subprocess.run(
        build_cmd, check=False, capture_output=True, text=True
    )
    if proc.returncode != 0:
        sys.stderr.write(proc.stderr)
        raise RuntimeError(
            f"ontology-build exited {proc.returncode}; "
            f"stderr printed above."
        )
    build_result = _json.loads(proc.stdout) if proc.stdout.strip() else {}
    # P2: assert the CLI reports the skip-flag status the
    # ``--skip-property-graph`` contract promises. Without
    # this, a CLI regression that returned success but a
    # different (or absent) status would silently pass the
    # JOBS_BY_PROJECT check in cell 1.5 too.
    assert build_result.get("property_graph_status") == "skipped:user_requested", (
        f"expected property_graph_status='skipped:user_requested', "
        f"got {build_result.get('property_graph_status')!r}"
    )
    print(f"property_graph_status = {build_result['property_graph_status']!r}")
    # P3 prep: surface ``rows_materialized`` so cell 1.5 can
    # prove the build actually populated base tables. A GQL
    # count comparison alone (after >= before) trivially
    # passes when both are zero — that would let an empty-
    # graph regression slip through.
    total_rows = sum(build_result.get("rows_materialized", {}).values())
    print(f"rows_materialized total = {total_rows}")


Discovered 3 sessions in agent_events
/usr/local/opt/python@3.13/bin/python3.13 -m bigquery_agent_analytics.cli ontology-build --project-id test-project-0728-467323 --dataset-id migration_v5_demo_56f9179f --ontology examples/migration_v5/ontology.yaml --binding examples/migration_v5/binding.yaml --session-ids 39593184-f408-4296-93af-09da35baeaed,b3f8823e-d228-4af3-9e9d-1e996609806b,bc44b2ba-1e98-445c-bd88-fd1a0bb3b10c --skip-property-graph --location US --format json


property_graph_status = 'skipped:user_requested'
rows_materialized total = 38


### 1.5 — verify the SDK never touched the graph DDL

Two-pronged evidence:

**(a)** Query `INFORMATION_SCHEMA.JOBS_BY_PROJECT` for `CREATE OR REPLACE PROPERTY
GRAPH` jobs targeting `mako_demo_graph` and carrying the SDK's
`sdk_feature='ontology-gql'` label, with `creation_time > before_skip_build_ts`.
Assert zero rows. Three filters — graph name, SDK label, post-cell-1.2 timestamp —
scope the check to DDL jobs the build itself issued. The graph-name filter (not a
dataset-name filter) catches regressions that would write the graph object into a
different dataset than the binding's; the label filter excludes user-authored DDL
even if a timestamp slipped.

**(b)** Re-run the GQL traversal. Assert the graph is still queryable and (now that
the base tables are populated) returns at least the same number of rows as the
before-build baseline.

In [12]:
if not FEATURES["skip_property_graph"]:
    print("Skipped: skip_property_graph feature off")
else:
    # JOBS_BY_PROJECT filter design mirrors the live #104
    # integration test
    # (``tests/test_integration_ontology_binding.py``):
    #
    #   1. ``creation_time > @before_ts`` — excludes the
    #      user's authored CREATE PROPERTY GRAPH from cell 1.2.
    #   2. ``UPPER(query) LIKE '%CREATE OR REPLACE PROPERTY GRAPH%'``
    #      — the spelling the SDK uses today.
    #   3. ``query LIKE @graph_name_pattern`` — the graph name
    #      lives in the DDL string regardless of which dataset
    #      a regressed SDK would target. Filtering by graph
    #      name (not by fully-qualified dataset path) catches
    #      the regression even in split orchestrator/binding-
    #      dataset setups where the SDK might write the graph
    #      into the orchestrator dataset rather than the
    #      binding's dataset.
    #   4. ``sdk_feature='ontology-gql'`` label — only SDK-
    #      issued property-graph jobs carry this label
    #      (``ontology_property_graph.py``). The user's
    #      authored DDL from cell 1.2 doesn't have it; even if
    #      the timestamp filter slipped, the label filter
    #      would still exclude it.
    #
    # The dataset-name filter from the previous draft was too
    # narrow — it would miss a regression that materialized
    # the graph into a different dataset.
    jobs_sql = rf"""
        SELECT job_id, query, creation_time
        FROM `region-{DATASET_LOCATION.lower()}`.INFORMATION_SCHEMA.JOBS_BY_PROJECT AS j
        WHERE creation_time > @before_ts
          AND UPPER(query) LIKE '%CREATE OR REPLACE PROPERTY GRAPH%'
          AND query LIKE @graph_name_pattern
          AND EXISTS (
            SELECT 1 FROM UNNEST(j.labels) AS l
            WHERE l.key = 'sdk_feature' AND l.value = 'ontology-gql'
          )
    """
    graph_name_pattern = "%mako_demo_graph%"
    offending = list(bq.query(
        jobs_sql,
        job_config=bigquery.QueryJobConfig(query_parameters=[
            bigquery.ScalarQueryParameter(
                "before_ts", "TIMESTAMP", before_skip_build_ts
            ),
            bigquery.ScalarQueryParameter(
                "graph_name_pattern", "STRING", graph_name_pattern
            ),
        ]),
    ).result())
    assert not offending, (
        f"Build issued {len(offending)} SDK-labelled "
        f"CREATE OR REPLACE PROPERTY GRAPH job(s) for "
        f"mako_demo_graph despite --skip-property-graph: "
        f"{[j.job_id for j in offending]}"
    )
    print(f"(a) No SDK-issued CREATE PROPERTY GRAPH job after {before_skip_build_ts.isoformat()}")

    # P3: assert the build actually materialized rows. Without
    # this, the GQL ``after >= before`` check below trivially
    # passes when both are zero — an empty-graph regression
    # would slip through.
    assert total_rows > 0, (
        f"Build reported rows_materialized total={total_rows}; "
        f"extraction silently returned an empty graph."
    )

    gql = f"""
        SELECT COUNT(*) AS n
        FROM GRAPH_TABLE(
          `{PROJECT_ID}.{DATASET_ID}.mako_demo_graph`
          MATCH (de:DecisionExecution)
          COLUMNS (de.decision_execution_id AS execution_id)
        )
    """
    after_skip_build_rows = next(iter(bq.query(gql).result())).n
    # (b) the graph definition the user authored is still
    # queryable, AND now reflects the materialized rows. Both
    # halves matter: ``>= before`` proves the object wasn't
    # destroyed; the ``total_rows > 0`` assertion above proves
    # the SDK populated the base tables the user's graph
    # definition reads from.
    assert after_skip_build_rows >= before_skip_build_rows, (
        f"GQL count went down: before={before_skip_build_rows}, "
        f"after={after_skip_build_rows} — the graph DDL may have been overwritten."
    )
    print(
        f"(b) GQL DecisionExecution count: "
        f"before={before_skip_build_rows}, after={after_skip_build_rows}"
    )


(a) No SDK-issued CREATE PROPERTY GRAPH job after 2026-05-14T16:05:18.390378+00:00


(b) GQL DecisionExecution count: before=0, after=2


Beat 1 is closed:

- The SDK populated the base tables (cell 1.4's `ontology-build` ran cleanly).
- The SDK did not issue any `CREATE PROPERTY GRAPH` job (cell 1.5(a)).
- The graph object you defined in cell 1.2 still answers GQL queries (cell 1.5(b)).

The graph DDL is yours — the SDK never touched it.

## Section 2 — Beat 2: pre-flight catches binding drift before you spend a dollar (#105)

**Guarantee:** Validate.

Bindings drift. Someone renames a column in BigQuery, forgets to update
`binding.yaml`, and the next `ontology-build` happily fires off `AI.GENERATE`
before realizing — extraction-time — that it can't find the column. The model call
already cost dollars by then.

`binding-validate` is the sub-second pre-flight: it loads ontology + binding, asks
BigQuery for each referenced table's actual schema, and reports any drift as
structured failures with `binding_path` + `bq_ref` pointing at the exact authoring
site. Exit code 1 → don't even start the build.

### 2.2 — inject column-rename drift

Rename `context_snapshot.snapshot_payload` to `snapshot_payload_v2`. The binding
still maps `ContextSnapshot.snapshotPayload → snapshot_payload`, so the live table
no longer has the column the binding expects.

This is exactly the kind of error someone introduces by hand — a one-line
`ALTER TABLE RENAME COLUMN` after the binding was already authored.

In [13]:
if not FEATURES["binding_validate"]:
    print("Skipped: binding_validate feature off")
else:
    rename_to = "snapshot_payload_v2"
    drift_sql = (
        f"ALTER TABLE `{PROJECT_ID}.{DATASET_ID}.context_snapshot` "
        f"RENAME COLUMN snapshot_payload TO {rename_to}"
    )
    print(drift_sql)
    bq.query(drift_sql).result()
    schema = bq.get_table(
        f"{PROJECT_ID}.{DATASET_ID}.context_snapshot"
    ).schema
    print("context_snapshot columns:", [f.name for f in schema])


ALTER TABLE `test-project-0728-467323.migration_v5_demo_56f9179f.context_snapshot` RENAME COLUMN snapshot_payload TO snapshot_payload_v2


context_snapshot columns: ['context_snapshot_id', 'snapshot_payload_v2', 'snapshot_timestamp', 'session_id', 'extracted_at']


### 2.3 — pre-flight catches the drift

`binding-validate` produces a structured report. Each failure carries:

- `code` — the failure class as a lowercase string: `missing_column`, `type_mismatch`,
  `missing_table`, `missing_dataset`, etc. (`f.code.value` from the SDK's `FailureCode`
  enum; the Python enum names are uppercase but the JSON values are lowercase.)
- `binding_element` — the binding entity or relationship name
  (e.g. `ContextSnapshot`)
- `binding_path` — indexed YAML path inside the binding, e.g.
  `binding.entities[2].properties[1].column`. The indices match the YAML's
  position-ordered structure so the path round-trips back to the source line.
- `bq_ref` — fully-qualified BigQuery reference (e.g.
  `proj.dataset.context_snapshot.snapshot_payload`)
- `detail` — a human-readable message

Exit code 1 means the build should not proceed.


In [14]:
if not FEATURES["binding_validate"]:
    print("Skipped: binding_validate feature off")
else:
    import json
    validate_cmd = [
        sys.executable, "-m", "bigquery_agent_analytics.cli",
        "binding-validate",
        "--project-id", PROJECT_ID,
        "--ontology", str(ONTOLOGY_PATH),
        "--binding", str(BINDING_PATH),
        "--location", DATASET_LOCATION,
        "--format", "json",
    ]
    drift_proc = subprocess.run(
        validate_cmd,
        check=False,             # exit 1 expected
        capture_output=True,
        text=True,
        env=child_env,
    )
    drift_report = json.loads(drift_proc.stdout) if drift_proc.stdout.strip() else {}
    # Print structured output BEFORE asserting so the user always
    # sees the report shape even if the cell raises. Cell 2.4's
    # restore is idempotent (it checks which column exists) so a
    # cell-2.3 failure does not leave the table in a permanently
    # broken state.
    print(f"exit code = {drift_proc.returncode}")
    print(f"report.ok = {drift_report.get('ok')}")
    print(f"failures  = {len(drift_report.get('failures', []))}")
    for f in drift_report.get("failures", []):
        print(
            f"  - {f['code']:<22s} "
            f"binding={f['binding_path']}  "
            f"bq={f['bq_ref']}"
        )
    # ``f.code.value`` from the SDK's ``FailureCode`` enum is
    # lowercase (``MISSING_COLUMN`` is the Python name; the JSON
    # value is ``"missing_column"``).
    assert drift_proc.returncode == 1, (
        f"expected exit 1 (failures present), got {drift_proc.returncode}"
    )
    assert any(
        f["code"] == "missing_column" for f in drift_report.get("failures", [])
    ), "expected at least one missing_column failure for the renamed column"


exit code = 1
report.ok = False
failures  = 1
  - missing_column         binding=binding.entities[2].properties[1].column  bq=test-project-0728-467323.migration_v5_demo_56f9179f.context_snapshot.snapshot_payload


### 2.4 — fix the drift and re-run

Restore the column name (in real life, you'd either rename back or update the
binding YAML — both fix the drift). Re-run `binding-validate`; assert
`report.ok == True`.

In [15]:
if not FEATURES["binding_validate"]:
    print("Skipped: binding_validate feature off")
else:
    import json
    # Local defaults so the cell stands alone after a kernel
    # restart that lost ``rename_to`` / ``validate_cmd`` from
    # cells 2.2 and 2.3.
    rename_to = locals().get("rename_to", "snapshot_payload_v2")
    if "validate_cmd" not in locals():
        validate_cmd = [
            sys.executable, "-m", "bigquery_agent_analytics.cli",
            "binding-validate",
            "--project-id", PROJECT_ID,
            "--ontology", str(ONTOLOGY_PATH),
            "--binding", str(BINDING_PATH),
            "--location", DATASET_LOCATION,
            "--format", "json",
        ]
    # Restore-or-noop. Even if cell 2.3 asserted before reaching
    # cell 2.4, this brings the table back to the binding-expected
    # shape so the rest of the notebook keeps working. The check
    # is intentionally column-presence-based, not state-flag-based,
    # because a partial run can leave the rename half-applied.
    current_columns = {
        f.name for f in bq.get_table(
            f"{PROJECT_ID}.{DATASET_ID}.context_snapshot"
        ).schema
    }
    if "snapshot_payload" in current_columns:
        print("Column already named snapshot_payload — no restore needed.")
    elif rename_to in current_columns:
        restore_sql = (
            f"ALTER TABLE `{PROJECT_ID}.{DATASET_ID}.context_snapshot` "
            f"RENAME COLUMN {rename_to} TO snapshot_payload"
        )
        print(restore_sql)
        bq.query(restore_sql).result()
    else:
        raise RuntimeError(
            f"context_snapshot has neither snapshot_payload nor "
            f"{rename_to}; columns={current_columns}"
        )

    ok_proc = subprocess.run(
        validate_cmd,
        check=False,
        capture_output=True,
        text=True,
        env=child_env,
    )
    ok_report = json.loads(ok_proc.stdout) if ok_proc.stdout.strip() else {}
    print(f"exit code = {ok_proc.returncode}")
    print(f"report.ok = {ok_report.get('ok')}")
    assert ok_proc.returncode == 0, (
        f"binding-validate should succeed after restore, "
        f"got exit {ok_proc.returncode}; stderr={ok_proc.stderr!r}"
    )
    assert ok_report.get("ok") is True, (
        f"report.ok should be True after restore; got {ok_report.get('ok')}"
    )


ALTER TABLE `test-project-0728-467323.migration_v5_demo_56f9179f.context_snapshot` RENAME COLUMN snapshot_payload_v2 TO snapshot_payload


exit code = 0
report.ok = True


### 2.5 — combine Beat 1 + Beat 2 in one `ontology-build`

Once the user owns the property graph (Beat 1) and the binding is pre-flighted
(Beat 2), the production-shape invocation collapses to a single call:

```
bq-agent-sdk ontology-build --skip-property-graph --validate-binding --ontology X --binding Y --session-ids ...
```

The build runs the pre-flight first; any failures short-circuit before
`AI.GENERATE` fires. Both `property_graph_status='skipped:user_requested'` and
`rows_materialized > 0` should appear in the output.

In [16]:
if not (FEATURES["binding_validate"] and FEATURES["skip_property_graph"]):
    print("Skipped: requires binding_validate + skip_property_graph features")
else:
    import json
    combined_cmd = [
        sys.executable, "-m", "bigquery_agent_analytics.cli",
        "ontology-build",
        "--project-id", PROJECT_ID,
        "--dataset-id", DATASET_ID,
        "--ontology", str(ONTOLOGY_PATH),
        "--binding", str(BINDING_PATH),
        "--session-ids", ",".join(session_ids),
        "--skip-property-graph",
        "--validate-binding",
        "--location", DATASET_LOCATION,
        "--format", "json",
    ]
    print(" ".join(combined_cmd))
    combined_proc = subprocess.run(
        combined_cmd,
        check=False,
        capture_output=True,
        text=True,
        env=child_env,
    )
    if combined_proc.returncode != 0:
        sys.stderr.write(combined_proc.stderr)
        raise RuntimeError(
            f"combined ontology-build exited "
            f"{combined_proc.returncode}; stderr printed above."
        )
    combined_result = json.loads(combined_proc.stdout)
    assert combined_result["property_graph_status"] == "skipped:user_requested", (
        f"expected property_graph_status='skipped:user_requested', "
        f"got {combined_result['property_graph_status']!r}"
    )
    combined_rows = sum(combined_result.get("rows_materialized", {}).values())
    assert combined_rows > 0, (
        f"combined build reported rows_materialized total={combined_rows}; "
        f"extraction silently returned empty."
    )
    print(f"property_graph_status = {combined_result['property_graph_status']!r}")
    print(f"rows_materialized total = {combined_rows}")


/usr/local/opt/python@3.13/bin/python3.13 -m bigquery_agent_analytics.cli ontology-build --project-id test-project-0728-467323 --dataset-id migration_v5_demo_56f9179f --ontology examples/migration_v5/ontology.yaml --binding examples/migration_v5/binding.yaml --session-ids 39593184-f408-4296-93af-09da35baeaed,b3f8823e-d228-4af3-9e9d-1e996609806b,bc44b2ba-1e98-445c-bd88-fd1a0bb3b10c --skip-property-graph --validate-binding --location US --format json


property_graph_status = 'skipped:user_requested'
rows_materialized total = 19


Beat 2 is closed:

- Drift was injected (cell 2.2) and the pre-flight caught it in under a second (cell 2.3).
- The fix loop is a single CLI call; assertions confirm `report.ok == True` after
  restore (cell 2.4).
- One end-to-end invocation combines the pre-flight + the skip-graph build (cell 2.5).

Extraction never started until the physical and logical worlds lined up.

## Section 3 — Beat 3: structured events extract deterministically; LLM only fills the gaps (#75 + #76)

**Guarantee:** Extract cheaply.

Today's ontology extraction is **session-aggregated `AI.GENERATE`**: one model call per
session, not one per event (`ontology_graph.py:100, 631`). The cost driver is
`sessions × tokens-per-session`, and tokens-per-session grows with the size of the
structured-event payloads in the transcript.

The four-guarantee story is that **compiled deterministic extractors handle the
structured part** (every `tool_call`'s typed input/output, every span whose payload
matches a registered event schema), and `AI.GENERATE` is only called for the
open-ended, narrative parts of the trace. Validation-gated pruning (per the [C2
decision](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/75#issuecomment-4363361603))
excludes spans whose compiled output passes the #76 validator from the transcript
the AI sees; partial-coverage spans stay in with a focused hint; validation-failed
spans fall back to AI entirely.

Two #75 sub-phases land independently:

| Sub-phase | What it ships | Section 3 cell |
|---|---|---|
| **C1** | `compile-extractors` harness + measurement report | 3.3 + 3.4 |
| **C2** | Runtime bundle loading + validation-gated prune | 3.5 + 3.7 |

Cells 3.3-3.5 + 3.7 require a **MAKO-specific reference extractor**
(`extract_mako_decision_event` analogous to the shipped
`extract_bka_decision_event`). PR #155 is fixtures-only — the reference extractor
lands in a follow-up PR (#156). For now those cells emit a `requires MAKO reference
extractor` skip marker; cell 3.2 (baseline cost view) and cell 3.6 (post-extract
`ValidationReport`) use real data already in `agent_events` and run end-to-end.

### 3.2 — baseline per-session transcript cost

Per-event token attribution is **not** available from current architecture: the AI
extraction query at `ontology_graph.py:90-136` returns only `(session_id, graph_json)`
and exposes no per-row usage metadata. The **session** is the cost unit.

This cell estimates per-session prompt size from the `content` payload of every
captured event. It's a **prompt-size estimate** (chars and an approximate 4-chars-
per-token model), not exact billing usage. Once #75 C2 ships the runtime bundle-
loading path, the same view will show the post-compile transcript size and the
delta is the savings story.

In [17]:
# Real data: query agent_events for per-session transcript size.
# ``content`` is JSON; ``LENGTH(TO_JSON_STRING(content))`` gives
# the wire-size in chars for each event. ``estimated_tokens`` uses
# the typical-English 4-chars-per-token rule of thumb — it's a
# ballpark for the demo, not a billing claim.
baseline_sql = f"""
    SELECT
      session_id,
      COUNT(*) AS event_count,
      SUM(COALESCE(LENGTH(TO_JSON_STRING(content)), 0)) AS transcript_chars,
      CAST(SUM(COALESCE(LENGTH(TO_JSON_STRING(content)), 0)) / 4.0 AS INT64)
          AS estimated_prompt_tokens
    FROM `{PROJECT_ID}.{DATASET_ID}.agent_events`
    WHERE session_id IS NOT NULL
    GROUP BY session_id
    ORDER BY transcript_chars DESC
"""
print(f"{'session_id':36s} {'events':>7s} {'chars':>8s} {'est_tokens':>11s}")
print("-" * 65)
baseline_rows = list(bq.query(baseline_sql).result())
for r in baseline_rows:
    print(
        f"{r.session_id:36s} {r.event_count:>7d} "
        f"{r.transcript_chars:>8d} {r.estimated_prompt_tokens:>11d}"
    )
total_chars = sum(r.transcript_chars for r in baseline_rows)
total_tokens = sum(r.estimated_prompt_tokens for r in baseline_rows)
print("-" * 65)
print(f"{'TOTAL':36s} {sum(r.event_count for r in baseline_rows):>7d} "
      f"{total_chars:>8d} {total_tokens:>11d}")
# Guard rails: a "live baseline" cell that prints zero rows
# or zero chars is the same as "Section 0 didn't actually
# populate agent_events" — surface that as a hard failure
# rather than letting the demo claim Beat 3 is live on an
# empty corpus. ``COALESCE(...,0)`` makes total_chars an int,
# so the comparison is safe.
assert baseline_rows, (
    "agent_events has zero session rows — re-run Section 0\'s "
    "agent driver before re-running this cell."
)
assert total_chars > 0, (
    f"every session\'s ``content`` is NULL or empty "
    f"(total_chars={total_chars}); the plugin captured rows "
    f"but none carry payloads."
)


session_id                            events    chars  est_tokens
-----------------------------------------------------------------


b3f8823e-d228-4af3-9e9d-1e996609806b      39    32832        8208
39593184-f408-4296-93af-09da35baeaed      40    29714        7429
bc44b2ba-1e98-445c-bd88-fd1a0bb3b10c      39    29002        7251
-----------------------------------------------------------------
TOTAL                                    118    91548       22888


### 3.3 — compile a deterministic extractor (C1)

`compile_extractor` takes a hand-written Python extractor source string, runs it
through every gate (AST safety check → smoke test against sample events → validator
→ fingerprint), and writes a bundle to disk. Re-compiling the same source against
the same inputs produces a byte-identical bundle.

Pairs with `measure_compile`, which runs the compiled extractor against a
**reference extractor** for parity and reports F1, per-event latency, fallback rate,
and an `attempt_failures` list for triage.

**Status:** This cell needs `examples/migration_v5/reference_extractor.py` to ship a
MAKO-specific reference extractor (`extract_mako_decision_event`). PR #155 is
fixtures-only; the reference extractor + this cell's live measurement land in a
follow-up PR (#156).

In [18]:
from pathlib import Path as _P
mako_ref_path = _P("examples/migration_v5/reference_extractor.py")
if not FEATURES["compiled_extractors_c1"]:
    print("Skipped: compiled_extractors_c1 feature off")
elif not mako_ref_path.exists():
    print(
        f"Skipped: requires MAKO reference extractor at "
        f"{mako_ref_path} — lands in follow-up PR #156."
    )
else:
    # Future-PR shape (PR #156). Documenting here so the
    # follow-up author has a target.
    from bigquery_agent_analytics.extractor_compilation import (
        compile_extractor,
        measure_compile,
    )
    # The follow-up will:
    #   1. Load extract_mako_decision_event from
    #      reference_extractor.py
    #   2. Hand-author a candidate compiled source.
    #   3. Pull ~100 ``tool_call`` events from agent_events
    #      as ``sample_events``.
    #   4. Call ``compile_extractor`` with the ontology +
    #      binding + sample events.
    #   5. Call ``measure_compile`` against the reference
    #      to get the parity report.
    # See ``tests/test_extractor_compilation_measurement.py``
    # for the integration-test shape.
    raise NotImplementedError(
        "PR #156: wire up reference_extractor.py + compile_extractor call"
    )


Skipped: requires MAKO reference extractor at examples/migration_v5/reference_extractor.py — lands in follow-up PR #156.


### 3.4 — fingerprint reproducibility

The compile loop emits `compile_fingerprint` (64-hex sha256) and
`compile_id = compile_fingerprint[:12]`. Re-compiling the same source + same inputs
produces the same fingerprint — that's the audit trail every extractor bundle
carries.

**Status:** Depends on 3.3 landing the live compile call.

In [19]:
if not FEATURES["compiled_extractors_c1"]:
    print("Skipped: compiled_extractors_c1 feature off")
elif not mako_ref_path.exists():
    print(
        "Skipped: requires the cell 3.3 compile call — "
        "lands in PR #156."
    )
else:
    # Re-run the same compile_extractor call and assert
    # the fingerprint matches. ``cache_hit=True`` on the
    # second call means the harness recognized the inputs
    # and short-circuited.
    raise NotImplementedError("PR #156: assert fingerprint reproducibility")


Skipped: requires the cell 3.3 compile call — lands in PR #156.


### 3.5 — runtime build with compiled bundles (C2)

C2 wires bundle loading into `ontology-build` so structured events are extracted by
the compiled deterministic path before `AI.GENERATE` sees them. Validation-gated:
the C2 wrapper consults the #76 validator on each compiled span and:

- **fully covered** spans → excluded from the AI transcript
- **partially covered** spans → AI sees them with a focused hint
- **validation failed** → fall back to the AI path

**Status:** Depends on cell 3.3's compile output. Lands with PR #156.

In [20]:
if not FEATURES["compiled_extractors_c2"]:
    print("Skipped: compiled_extractors_c2 feature off")
elif not mako_ref_path.exists():
    print("Skipped: requires the cell 3.3 bundle — lands in PR #156.")
else:
    # Re-run ontology-build with bundles loaded; pull the
    # post-prune per-session token estimate from
    # agent_events; compute the delta against the baseline
    # captured in cell 3.2.
    raise NotImplementedError("PR #156: ontology-build with compiled bundles")


Skipped: requires the cell 3.3 bundle — lands in PR #156.


### 3.6 — `ValidationReport` from #76: FIELD / NODE / EDGE scopes

Independent of #75 (no compile needed): the SDK ships a graph validator that
classifies every failure by the **smallest safe replacement unit** (`FallbackScope`).
A FIELD-scope failure can be patched by re-extracting one property; a NODE failure
drops the node + dependent edges; an EDGE failure drops just that edge. The scope
is what makes per-field AI fallback safe — the SDK only re-spends `AI.GENERATE` on
the slice that actually broke.

This cell hand-builds an `ExtractedGraph` designed to fail in each scope and prints
the structured report. Synthetic fixtures — no live BigQuery needed.

In [21]:
if not FEATURES["validate_extracted_graph"]:
    print("Skipped: validate_extracted_graph feature off")
else:
    from bigquery_ontology import (
        load_ontology as _load_ontology,
        load_binding as _load_binding,
    )
    from bigquery_agent_analytics.extracted_models import (
        ExtractedGraph, ExtractedNode, ExtractedEdge, ExtractedProperty,
    )
    from bigquery_agent_analytics.graph_validation import (
        validate_extracted_graph_from_ontology, FallbackScope,
    )

    _ont = _load_ontology(str(ONTOLOGY_PATH))
    _bind = _load_binding(str(BINDING_PATH), ontology=_ont)

    # Synthetic graph designed to fail in each scope:
    #   - NODE: ``UnknownEntity`` not declared in the
    #     ontology.
    #   - FIELD: ``DecisionExecution.latencyMs`` is INT64
    #     per MAKO TTL; a string value fails type check.
    #   - EDGE: ``nonExistentEdge`` not a declared
    #     relationship.
    synthetic = ExtractedGraph(
        name="mako_demo_graph",
        nodes=[
            ExtractedNode(
                node_id="sess:UnknownEntity:id=u1",
                entity_name="UnknownEntity",
            ),
            ExtractedNode(
                node_id=(
                    "sess:DecisionExecution:id=exec-1,"
                    "decision_execution_id=exec-1"
                ),
                entity_name="DecisionExecution",
                properties=[
                    ExtractedProperty(name="id", value="exec-1"),
                    ExtractedProperty(name="latencyMs", value="not-a-number"),
                ],
            ),
            ExtractedNode(
                node_id=(
                    "sess:DecisionPoint:id=dp-1,"
                    "decision_point_id=dp-1"
                ),
                entity_name="DecisionPoint",
                properties=[ExtractedProperty(name="id", value="dp-1")],
            ),
        ],
        edges=[
            ExtractedEdge(
                edge_id="e-bad",
                relationship_name="nonExistentEdge",
                from_node_id="sess:DecisionExecution:id=exec-1",
                to_node_id="sess:DecisionPoint:id=dp-1",
            ),
        ],
    )
    report = validate_extracted_graph_from_ontology(_ont, _bind, synthetic)
    print(f"failures: {len(report.failures)}  (report.ok = {report.ok})\n")
    for scope in (FallbackScope.FIELD, FallbackScope.NODE, FallbackScope.EDGE):
        matches = report.by_scope(scope)
        print(f"  {scope.value.upper():5s}: {len(matches)} failure(s)")
        for f in matches:
            print(f"    - {f.code:25s} at {f.path}")
    # Asserting each scope fired catches regressions in
    # the validator's scope classification (which is what
    # makes #76's per-field AI fallback safe to ship).
    assert any(f.scope is FallbackScope.NODE for f in report.failures), (
        "expected at least one NODE failure"
    )
    assert any(f.scope is FallbackScope.FIELD for f in report.failures), (
        "expected at least one FIELD failure"
    )
    assert any(f.scope is FallbackScope.EDGE for f in report.failures), (
        "expected at least one EDGE failure"
    )
    print("\nNODE + FIELD + EDGE all fired — scope classification works.")


failures: 3  (report.ok = False)

  FIELD: 1 failure(s)
    - type_mismatch             at nodes[1].properties[1].value
  NODE : 1 failure(s)
    - unknown_entity            at nodes[0].entity_name
  EDGE : 1 failure(s)
    - unknown_relationship      at edges[0].relationship_name

NODE + FIELD + EDGE all fired — scope classification works.


### 3.7 — savings table (placeholder)

The full savings story is `(session_id, transcript_chars_before, transcript_chars_after, estimated_prompt_tokens_before, estimated_prompt_tokens_after, compiled_spans, partial_fallback_count, full_fallback_count)` plus a job-level row
from `INFORMATION_SCHEMA.JOBS_BY_PROJECT` (`total_bytes_processed`, `total_slot_ms`,
duration).

**Status:** Requires the C2 runtime build from cell 3.5 to produce post-prune
numbers. Cell 3.2's baseline is half the picture; the delta lands with PR #156.

Beat 3 status:

- The cost framing (3.1) and the baseline per-session view (3.2) are live and run
  against the real `agent_events` we populated in Section 0.
- The post-extract `ValidationReport` (3.6) runs end-to-end against synthetic
  fixtures: NODE + FIELD + EDGE scopes all fire as expected. That's the #76
  guarantee that makes per-field AI fallback safe.
- The compile + runtime + savings cells (3.3, 3.4, 3.5, 3.7) wait on a MAKO-specific
  reference extractor. The shipped `extract_bka_decision_event` is for BKA's domain;
  the MAKO equivalent (`extract_mako_decision_event` against the agent's `tool_call`
  event payloads) is the natural next artifact. PR #156 will land it + flip those
  cells live.

## Section 4 — Beat 4: user-typed inputs resolve to canonical concepts (#58)

**Guarantee:** Resolve.

The graph is populated. The user wants to ask questions in their own words —
`"audience segment"`, `"creative variant"`, `"DecisionExecution"` — but the property
graph indexes nodes by canonical entity names (`Candidate`, `DecisionPoint`,
or its canonical entity name like `DecisionExecution`). Beat 4 closes that gap.

The pipeline:

1. **Emit a concept index** at compile time via `gm compile --emit-concept-index`
   (shipped in [PR #92](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/pull/92)).
   Two atomic-per-statement tables — the index itself plus a `__meta` sibling —
   keyed on a shared `compile_fingerprint`.
2. **Read it at runtime** via `OntologyRuntime` + `LabelSynonymResolver` (shipped
   in #58). The reader queries the index by `label`, optionally filtered by
   `label_kind` (`name` / `pref` / `alt` / `hidden` / `synonym` / `notation`) and language.

The demo's MAKO ontology was imported from a TTL that doesn't ship explicit
`skos:altLabel` / `skos:prefLabel` rows, so the concept index has one row per entity
with `label_kind='name'`. The resolver API surface is identical when richer labels
are present (e.g. a TTL with `mako:Candidate skos:altLabel "audience segment"@en` would
add a second row whose `label_kind='alt'`); the demo resolves `"DecisionExecution"`
→ `DecisionExecution` to keep the example concrete with the data we have.

### 4.2 — emit the concept index

`gm compile --emit-concept-index --concept-index-table <FQN>` appends two
`CREATE OR REPLACE TABLE` statements to the property-graph DDL output. The index
row schema:

```
entity_name STRING, label STRING, label_kind STRING,
notation STRING, scheme STRING, language STRING,
is_abstract BOOL,
compile_id STRING, compile_fingerprint STRING
```

`compile_fingerprint` is the 64-hex sha256 over the (ontology, binding,
compiler-version, target) tuple; `compile_id = compile_fingerprint[:12]`. The two
tables share `compile_fingerprint` so a reader can tell whether the index it loaded
and the `__meta` row it loaded come from the same compile.

In [22]:
if not FEATURES["concept_index_reader"]:
    print("Skipped: concept_index_reader feature off")
else:
    CONCEPT_INDEX_TABLE = (
        f"{PROJECT_ID}.{DATASET_ID}.mako_concept_index"
    )
    compile_cmd = [
        sys.executable, "-m", "bigquery_ontology.cli", "compile",
        "--emit-concept-index",
        "--concept-index-table", CONCEPT_INDEX_TABLE,
        "--ontology", str(ONTOLOGY_PATH),
        str(BINDING_PATH),
    ]
    print(" ".join(compile_cmd))
    compile_out = subprocess.check_output(
        compile_cmd, text=True, env=child_env
    )
    # Extract just the two ``CREATE OR REPLACE TABLE``
    # statements so the cell's output isn't dominated by
    # the property-graph DDL we already showed in Beat 1.
    ci_start = compile_out.find("CREATE OR REPLACE TABLE")
    assert ci_start >= 0, (
        "no CREATE OR REPLACE TABLE in compile output; "
        "concept-index emission may have failed."
    )
    concept_index_ddl = compile_out[ci_start:]
    # Print first ~20 lines so the structure is visible
    # without flooding the notebook.
    preview = concept_index_ddl.splitlines()
    for line in preview[:25]:
        print(line)
    if len(preview) > 25:
        print(f"... ({len(preview) - 25} more lines)")


/usr/local/opt/python@3.13/bin/python3.13 -m bigquery_ontology.cli compile --emit-concept-index --concept-index-table test-project-0728-467323.migration_v5_demo_56f9179f.mako_concept_index --ontology examples/migration_v5/ontology.yaml examples/migration_v5/binding.yaml


CREATE OR REPLACE TABLE `test-project-0728-467323.migration_v5_demo_56f9179f.mako_concept_index`
AS SELECT * FROM UNNEST(ARRAY<STRUCT<entity_name STRING, label STRING, label_kind STRING, notation STRING, scheme STRING, language STRING, is_abstract BOOL, compile_id STRING, compile_fingerprint STRING>>[
  ('AgentSession', 'AgentSession', 'name', NULL, NULL, NULL, FALSE, 'ba9c1ff85870', 'ba9c1ff858707439dd96dfe9bbf66814a67bb7fb8b985213b44ecba5d28a7303'),
  ('Candidate', 'Candidate', 'name', NULL, NULL, NULL, FALSE, 'ba9c1ff85870', 'ba9c1ff858707439dd96dfe9bbf66814a67bb7fb8b985213b44ecba5d28a7303'),
  ('ContextSnapshot', 'ContextSnapshot', 'name', NULL, NULL, NULL, FALSE, 'ba9c1ff85870', 'ba9c1ff858707439dd96dfe9bbf66814a67bb7fb8b985213b44ecba5d28a7303'),
  ('DecisionExecution', 'DecisionExecution', 'name', NULL, NULL, NULL, FALSE, 'ba9c1ff85870', 'ba9c1ff858707439dd96dfe9bbf66814a67bb7fb8b985213b44ecba5d28a7303'),
  ('DecisionPoint', 'DecisionPoint', 'name', NULL, NULL, NULL, FALSE, 'ba9c

### 4.3 — apply the index + resolve a user-typed label

Apply the `CREATE OR REPLACE TABLE` statements to the scratch dataset, then
construct `OntologyRuntime` with `concept_index_table` set. `LabelSynonymResolver`
wraps the runtime's `concept_index` lookup; `resolver.resolve(query)` returns
ranked `ResolverCandidate` rows.

The candidate carries:
- `entity_name` — canonical entity
- `matched_label` / `matched_label_kind` — what the index matched against
  (`name` / `pref` / `alt` / `hidden` / `synonym` / `notation`)
- `compile_id` / `compile_fingerprint` — provenance: which compile produced
  the index this row came from

In [23]:
if not FEATURES["concept_index_reader"]:
    print("Skipped: concept_index_reader feature off")
else:
    # Apply the two CREATE OR REPLACE TABLE statements.
    # The compile output may contain multiple statements
    # separated by ``;\n``; running them one at a time keeps
    # the queries small and the error messages clear.
    for stmt in concept_index_ddl.split(";"):
        stmt = stmt.strip()
        if not stmt:
            continue
        bq.query(stmt + ";").result()
    print(f"Concept index materialized to {CONCEPT_INDEX_TABLE}")

    # Read the compiler version straight from the __meta
    # row the compiler just emitted. Hard-coding a literal
    # (``"bigquery_ontology 0.2.2"``) would drift the moment
    # the package version bumps: the compile_fingerprint
    # closes over the version string, so any mismatch
    # between what was emitted and what the runtime is told
    # trips FingerprintMismatchError. The __meta row is the
    # SDK's own authoritative answer.
    meta_row = next(iter(bq.query(
        f"SELECT compiler_version, compile_fingerprint "
        f"FROM `{CONCEPT_INDEX_TABLE}__meta`"
    ).result()))
    COMPILER_VERSION = meta_row.compiler_version
    print(f"compiler_version    = {COMPILER_VERSION!r}")
    print(f"compile_fingerprint = {meta_row.compile_fingerprint}")

    from bigquery_agent_analytics.ontology_runtime import (
        OntologyRuntime, LabelSynonymResolver,
    )
    runtime = OntologyRuntime.from_files(
        ontology_path=str(ONTOLOGY_PATH),
        binding_path=str(BINDING_PATH),
        compiler_version=COMPILER_VERSION,
        concept_index_table=CONCEPT_INDEX_TABLE,
        bq_client=bq,
    )
    resolver = LabelSynonymResolver(runtime)

    # Resolve a user-typed query against the canonical
    # entity names. With a richer TTL, the same call
    # resolves ``"audience segment"`` → ``Candidate``
    # (via skos:altLabel) — only the underlying ontology
    # changes, the resolver call doesn't.
    user_query = "DecisionExecution"
    candidates = resolver.resolve(user_query, limit=5)
    print(f"resolver.resolve({user_query!r}) → {len(candidates)} candidate(s)")
    for c in candidates:
        print(
            f"  entity={c.entity_name:24s} "
            f"label={c.matched_label!r} kind={c.matched_label_kind} "
            f"compile_id={c.compile_id}"
        )
    assert candidates, (
        f"expected at least one candidate for {user_query!r}; "
        f"concept-index may not have been applied."
    )
    # Lock the canonical entity for cell 4.4's GQL.
    resolved_entity = candidates[0].entity_name
    resolved_compile_id = candidates[0].compile_id


Concept index materialized to test-project-0728-467323.migration_v5_demo_56f9179f.mako_concept_index


compiler_version    = 'bigquery_ontology 0.2.2'
compile_fingerprint = ba9c1ff858707439dd96dfe9bbf66814a67bb7fb8b985213b44ecba5d28a7303


resolver.resolve('DecisionExecution') → 1 candidate(s)
  entity=DecisionExecution        label='DecisionExecution' kind=name compile_id=ba9c1ff85870


### 4.4 — GQL traversal scoped by the resolved entity

Once `resolver.resolve(user_query)` returns a canonical `entity_name`, the GQL
traversal plugs it in as the `MATCH` label. Below: count `DecisionExecution`
nodes for the sessions Section 0 populated. The follow-up hub-shape sample
(`(DecisionExecution)-[partOfSession]->(AgentSession)`) is deferred to PR #156:
MAKO's AI extraction in this demo doesn't synthesize `AgentSession` nodes from
the plugin envelope, so the hub edge has zero materialized rows. The count
traversal alone proves the resolver → graph wiring works end-to-end.

This is the climax query: user-typed label (canonical in this fixture; natural-language `skos:altLabel` / `skos:prefLabel` rows in richer ontologies) → resolver returns a canonical entity → traversal works against the property graph the user authored.

In [24]:
if not FEATURES["concept_index_reader"]:
    print("Skipped: concept_index_reader feature off")
else:
    # The resolved entity is a string, but GoogleSQL's
    # ``GRAPH_TABLE`` MATCH label is a static identifier,
    # not a parameter — so build the SQL via f-string after
    # asserting the resolved entity is in the demo allowlist.
    # The check is defence-in-depth: ``LabelSynonymResolver``
    # only returns rows from the SDK-emitted concept index,
    # which is bounded by the ontology, but a future
    # multi-ontology setup might need stricter scoping.
    assert resolved_entity in {
        "AgentSession", "Candidate", "ContextSnapshot",
        "DecisionExecution", "DecisionPoint", "SelectionOutcome",
    }, f"resolved entity {resolved_entity!r} outside demo allowlist"

    # Entity → PK column map (matches binding.yaml). The GQL
    # MATCH label is a static identifier, but the COLUMNS
    # accessor needs the entity-specific PK column name —
    # ``id`` no longer exists on the underlying table since
    # ``make_binding`` renamed the PK column per-entity.
    _PK_COL = {
        "AgentSession": "agent_session_id",
        "Candidate": "candidate_id",
        "ContextSnapshot": "context_snapshot_id",
        "DecisionExecution": "decision_execution_id",
        "DecisionPoint": "decision_point_id",
        "SelectionOutcome": "selection_outcome_id",
    }
    pk_col = _PK_COL[resolved_entity]

    count_gql = f"""
        SELECT COUNT(*) AS n
        FROM GRAPH_TABLE(
          `{PROJECT_ID}.{DATASET_ID}.mako_demo_graph`
          MATCH (n:{resolved_entity})
          COLUMNS (n.{pk_col} AS id)
        )
    """
    count = next(iter(bq.query(count_gql).result())).n
    print(
        f"GRAPH_TABLE COUNT on {resolved_entity} "
        f"(compile_id={resolved_compile_id}) = {count}"
    )
    # The resolver-to-graph wiring is only meaningful if the
    # resolved entity has rows in the base tables. A zero
    # count means either Section 0 populated agent_events
    # but Beat 1 didn't materialize rows for this entity, or
    # the resolver returned an entity we materialize zero of
    # — either way the demo's climax claim ("user-typed
    # name routes to real graph data") would be empty.
    assert count > 0, (
        f"GRAPH_TABLE matched zero {resolved_entity} nodes; "
        f"the build in Beat 1 may have produced an empty graph."
    )

    # Hub-shape traversal: for each DecisionExecution
    # node, follow ``partOfSession`` to the AgentSession
    # it belongs to. In this demo the AI extraction
    # doesn't synthesize ``AgentSession`` nodes from the
    # plugin envelope, so the edge has zero materialized
    # rows; PR #156 lands the envelope→graph synthesis.
    # The block below degrades gracefully to a deferred-
    # path note rather than asserting.
    if resolved_entity == "DecisionExecution":
        hub_gql = f"""
            SELECT
              de_id, s_id
            FROM GRAPH_TABLE(
              `{PROJECT_ID}.{DATASET_ID}.mako_demo_graph`
              MATCH (de:DecisionExecution)-[:partOfSession]->(s:AgentSession)
              COLUMNS (de.decision_execution_id AS de_id, s.agent_session_id AS s_id)
            )
            LIMIT 5
        """
        hub_rows = list(bq.query(hub_gql).result())
        print()
        if not hub_rows:
            # AI extraction in this demo extracts the four
            # decision-flow entities the agent's tools return
            # (DecisionExecution / DecisionPoint / Candidate /
            # ContextSnapshot / SelectionOutcome). It does
            # *not* synthesize an AgentSession node + a
            # ``partOfSession`` edge because the agent's
            # tools don't return a session-shaped payload —
            # the ``session_id`` lives in the plugin envelope.
            # A future ``mako_artifacts`` extension could
            # synthesize the envelope-side node + edge as a
            # post-extraction pass; for the demo, the
            # heterogeneous-edge GQL count above is sufficient
            # to prove "resolver → real graph data."
            print(
                "Hub-shape (DecisionExecution -[partOfSession]->\n"
                "AgentSession) returned zero rows. AgentSession\n"
                "synthesis from the plugin envelope is a follow-up\n"
                "(PR #156); see Beat 4 closing for the design note."
            )
        else:
            print(f"{'decision_execution_id':38s} {'agent_session_id':38s}")
            print("-" * 78)
            for row in hub_rows:
                print(f"{row.de_id:38s} {row.s_id:38s}")


GRAPH_TABLE COUNT on DecisionExecution (compile_id=ba9c1ff85870) = 3



Hub-shape (DecisionExecution -[partOfSession]->
AgentSession) returned zero rows. AgentSession
synthesis from the plugin envelope is a follow-up
(PR #156); see Beat 4 closing for the design note.


Beat 4 is closed:

- The SDK emitted a concept index alongside the property graph at compile time
  (cell 4.2). Each row carries `compile_id` + `compile_fingerprint` for
  provenance.
- The user-typed query resolved to a canonical entity via `LabelSynonymResolver`
  (cell 4.3). The resolver returns the same `compile_fingerprint` it loaded the
  index with — drift between the compile that produced the graph and the compile
  that produced the index would surface as a mismatched fingerprint.
- The resolved canonical name plugs straight into a GQL traversal against the
  user-authored property graph (cell 4.4).
- The hub-shape variant of that traversal (`DecisionExecution -[partOfSession]->
  AgentSession`) currently returns zero rows because the AI extraction in Beat 1
  draws only on what the agent's tools return — and the tools don't return the
  envelope-side ``session_id``. Synthesizing the ``AgentSession`` node + the
  ``partOfSession`` edge from the plugin envelope is a natural follow-up
  (lands with PR #156).

In this fixture the user typed the canonical entity label (`DecisionExecution`)
because the MAKO TTL ships no SKOS labels — the concept index has one row per
entity with `label_kind='name'`. The same resolver call —
`LabelSynonymResolver.resolve(query)` — handles richer ontologies that emit
`skos:altLabel` / `skos:prefLabel` / `skos:notation` rows: the resolver picks up
the additional rows automatically and re-ranks by label kind
(`name` > `pref` > `alt` > `hidden` > `synonym` > `notation`). Only the ontology changes;
the notebook code doesn't.

## Section 5 — close

### Four-guarantee recap

| Guarantee | Before | After |
|---|---|---|
| **Own** | `CREATE OR REPLACE PROPERTY GRAPH` every build (SDK overwrites your DDL) | `--skip-property-graph`; you own the graph object, the SDK populates base tables. |
| **Validate** | First failure is at extraction time, after `AI.GENERATE` already ran | Sub-second pre-flight against live BigQuery schema before extraction starts. |
| **Extract cheaply** | `AI.GENERATE` over the full transcript every session | Compiled deterministic extractors handle structured events; AI fallback only for semantic gaps. Cell 3.2 establishes the pre-compile baseline; the compiled-savings delta lands with PR #156. |
| **Resolve** | User types an entity label and the GQL query depends on knowing its canonical name | SKOS concept-index lookup resolves user-typed inputs to canonical names before GQL runs. In this fixture the input is the canonical entity name; richer ontologies that ship `skos:altLabel` / `skos:prefLabel` rows resolve natural-language inputs via the same call. |

### Read more

Each guarantee has its own issue + landing PRs:

- **Own** — [#104](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/104)
- **Validate** — [#105](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/105) (pre-flight) + [#76](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/76) (post-extract `ValidationReport`)
- **Extract cheaply** — [#75](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/75) (compile-extractors C1 + C2)
- **Resolve** — [#58](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/58) (concept-index reader; emission shipped in [PR #92](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/pull/92))
- **Storyboard** — [#107](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/107) (this notebook's cell-by-cell plan)

### What's next

- A default `@builtin:adk-events` ontology so users with no domain-specific extraction needs can run the SDK with **zero authored YAML**.
- Phase 2 of [#75](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/75): session-aggregated compilation that reduces per-session AI cost further.
- Additional resolver layers in [#58](https://github.com/GoogleCloudPlatform/BigQuery-Agent-Analytics-SDK/issues/58) beyond `LabelSynonymResolver` (semantic similarity, embedding-based).